In [2]:
import os
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, DynamicCache


In [3]:
!pip install -U bitsandbytes>=0.46.1

In [4]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-80GB


In [19]:
PROMPT_ID = "union_find"
PROMPT_TEXT = "Write a Python code to implement union find. ONLY output the code - no explanations or markdown."

DRAFT_MODEL_ID = "meta-llama/Llama-3.2-1B"
TARGET_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
HF_TOKEN = "hf_%"

MAX_NEW_TOKENS = 256
GAMMA = 4
DRAFT_ON_GPU = True         # keep TinyLlama on CPU if GPU memory is tight
USE_4BIT_TARGET = True      # very helpful on Colab for Llama-3.1-8B


In [6]:
!pip -q install transformers accelerate bitsandbytes sentencepiece

import os
import time
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# =========================
# Config
# =========================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

# =========================
# Load tokenizers
# =========================
target_tok = AutoTokenizer.from_pretrained(TARGET_MODEL_ID, token=HF_TOKEN)
draft_tok = AutoTokenizer.from_pretrained(DRAFT_MODEL_ID, token=HF_TOKEN)

if target_tok.pad_token_id is None:
    target_tok.pad_token_id = target_tok.eos_token_id
if draft_tok.pad_token_id is None:
    draft_tok.pad_token_id = draft_tok.eos_token_id


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [7]:
# =========================
# Load models
# =========================
quant_config = None
if USE_4BIT_TARGET and DEVICE == "cuda":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=TARGET_DTYPE,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_ID,
    token=HF_TOKEN,
    device_map="auto" if DEVICE == "cuda" else None,
    torch_dtype=TARGET_DTYPE if quant_config is None else None,
    quantization_config=quant_config,
)
target_model.eval()

if DRAFT_ON_GPU and DEVICE == "cuda":
    draft_model = AutoModelForCausalLM.from_pretrained(
        DRAFT_MODEL_ID,
        token=HF_TOKEN,
        torch_dtype=TARGET_DTYPE,
        device_map="auto",
    )
else:
    draft_model = AutoModelForCausalLM.from_pretrained(
        DRAFT_MODEL_ID,
        token=HF_TOKEN,
        torch_dtype=torch.float32 if DEVICE == "cpu" else TARGET_DTYPE,
    )
    draft_model.to("cpu" if not DRAFT_ON_GPU else DEVICE)
draft_model.eval()

print("Target device map / device loaded.")
print("Target model loaded.")
print("Draft model loaded.")



model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Target device map / device loaded.
Target model loaded.
Draft model loaded.


In [ ]:
print(target_tok.__class__)
print(draft_tok.__class__)
print(target_tok.vocab_size, draft_tok.vocab_size)
print(target_tok.encode("def binary_search(arr, target):"))
print(draft_tok.encode("def binary_search(arr, target):"))

<class 'transformers.tokenization_utils_tokenizers.TokenizersBackend'>
<class 'transformers.tokenization_utils_tokenizers.TokenizersBackend'>
128000 128000
[128000, 755, 8026, 10947, 11179, 11, 2218, 1680]
[128000, 755, 8026, 10947, 11179, 11, 2218, 1680]


In [15]:
# =========================
# Helpers
# =========================
def maybe_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def verify_with_target_once_cached(target_past, target_next_logits, proposed_ids):
    """
    Invariant on entry:
      - target_past covers positions [0 .. P-1]
      - target_next_logits = target distribution for position P  shape [1, vocab]
      - proposed_ids = draft tokens for positions [P .. P+gamma-1]  shape [1, gamma]

    What we do:
      - Run target on proposed_ids (positions P..P+gamma-1)
      - out.logits[:, i, :] = target distribution for position P+i+1
      - So target's verdict on proposed_ids[:, i] comes from:
          i=0: target_next_logits         (predicts pos P,   checks proposed_ids[:,0])
          i=1: out.logits[:, 0, :]        (predicts pos P+1, checks proposed_ids[:,1])
          i=k: out.logits[:, k-1, :]      (predicts pos P+k, checks proposed_ids[:,k])

    Returns:
      accepted_len      - how many draft tokens the target agrees with
      correction_id     - target's token at first mismatch (None if fully accepted)
      new_target_past   - cache covering [0 .. P+gamma-1]
      new_target_logits - target distribution for position P+gamma (next round)
    """
    assert proposed_ids.dim() == 2 and proposed_ids.shape[0] == 1
    prop_len = proposed_ids.shape[1]

    if prop_len == 0:
        return 0, None, target_past, target_next_logits

    device = next(target_model.parameters()).device

    with torch.no_grad():
        out = target_model(
            input_ids=proposed_ids.to(device),
            past_key_values=target_past,
            use_cache=True,
        )
    # out.logits shape: [1, prop_len, vocab]
    # out.logits[:, i, :] = distribution for position P+i+1
    #                      = verdict on proposed_ids[:, i+1]
    # target_next_logits   = distribution for position P
    #                      = verdict on proposed_ids[:, 0]

    # Build target verdicts aligned with proposed_ids[:, 0..prop_len-1]
    # verdict[i] = what target predicts at position P+i = argmax of:
    #   i=0 -> target_next_logits
    #   i>0 -> out.logits[:, i-1, :]
    all_logits = torch.cat(
        [target_next_logits.unsqueeze(1), out.logits[:, :-1, :]], dim=1
    )  # shape [1, prop_len, vocab]
    target_preds = torch.argmax(all_logits, dim=-1)[0]  # shape [prop_len]

    proposed_flat = proposed_ids[0].to(target_preds.device)

    accepted_len  = 0
    correction_id = None
    for i in range(prop_len):
        if int(target_preds[i].item()) == int(proposed_flat[i].item()):
            accepted_len += 1
        else:
            correction_id = int(target_preds[i].item())
            break

    # new_target_logits = distribution for position P+prop_len (next round's starting point)
    new_target_logits = out.logits[:, -1, :]

    return accepted_len, correction_id, out.past_key_values, new_target_logits

In [13]:
def speculative_decode(prompt_text: str, max_new_tokens: int = 128, gamma: int = 4):
    """
    Minimal, correct speculative decoding.

    Core invariant: at the start of every iteration,
      - current_ids = full sequence so far (prompt + all accepted tokens)
      - target_past = KV cache for current_ids[:-1] ... NO.

    Simplest correct approach: NO incremental cache tricks between rounds.
    Just use cache within a round, rebuild at round boundaries.
    This is slower but CORRECT. We optimize after.
    """
    messages = [{"role": "user", "content": prompt_text}]
    formatted_prompt = target_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

    target_device = next(target_model.parameters()).device
    draft_device  = next(draft_model.parameters()).device

    input_ids  = target_tok(formatted_prompt, return_tensors="pt")["input_ids"]
    prompt_len = input_ids.shape[1]
    current_ids = input_ids.to(target_device)

    generated_target_tokens = 0
    accepted_tokens  = 0
    proposed_tokens  = 0
    rejection_events = 0

    while generated_target_tokens < max_new_tokens:
        remaining  = max_new_tokens - generated_target_tokens
        step_gamma = min(gamma, remaining)

        # ── 1. Draft: run on full current_ids, then autoregressively
        #            generate step_gamma tokens. Cache only within this block.
        with torch.no_grad():
            draft_out = draft_model(
                input_ids=current_ids.to(draft_device),
                use_cache=True,
            )
        d_past    = draft_out.past_key_values
        d_logits  = draft_out.logits[:, -1:, :]          # [1, 1, vocab]
        d_next    = torch.argmax(d_logits, dim=-1)        # [1, 1]

        proposals = [d_next]
        with torch.no_grad():
            for _ in range(step_gamma - 1):
                d_out2 = draft_model(
                    input_ids=proposals[-1].to(draft_device),
                    past_key_values=d_past,
                    use_cache=True,
                )
                d_past = d_out2.past_key_values
                proposals.append(torch.argmax(d_out2.logits[:, -1:, :], dim=-1))

        # proposed_ids[0, i] = draft proposal for position (current_len + i)
        proposed_ids = torch.cat(proposals, dim=1).to(target_device)  # [1, gamma]
        proposed_tokens += step_gamma

        # ── 2. Target: run on current_ids + proposed_ids in one forward pass.
        #    NO cache from previous round — rebuild fresh every time.
        #    This is the only way to guarantee correctness.
        verify_input = torch.cat([current_ids, proposed_ids], dim=1)  # [1, cur+gamma]

        with torch.no_grad():
            t_out = target_model(
                input_ids=verify_input.to(target_device),
                use_cache=False,   # don't cache — we rebuild each round
            )

        # t_out.logits shape: [1, cur+gamma, vocab]
        # t_out.logits[:, i, :] = target distribution for position i+1
        # So to verify proposed_ids[:, j] (at position cur+j):
        #   we need t_out.logits[:, cur-1+j, :]
        cur_len = current_ids.shape[1]

        # target_preds[j] = what target predicts at position cur+j
        # = argmax of t_out.logits[:, cur-1+j, :]
        # j = 0..gamma-1
        target_logits_for_proposals = t_out.logits[:, cur_len-1 : cur_len-1+step_gamma, :]
        # shape [1, gamma, vocab]
        target_preds = torch.argmax(target_logits_for_proposals, dim=-1)[0]  # [gamma]
        proposed_flat = proposed_ids[0]  # [gamma]

        # ── 3. Find acceptance length
        accepted_len  = 0
        correction_id = None
        for i in range(step_gamma):
            if int(target_preds[i].item()) == int(proposed_flat[i].item()):
                accepted_len += 1
            else:
                correction_id = int(target_preds[i].item())
                break

        # ── 4. Update current_ids
        if accepted_len > 0:
            current_ids = torch.cat(
                [current_ids, proposed_ids[:, :accepted_len]], dim=1
            )
            generated_target_tokens += accepted_len
            accepted_tokens         += accepted_len

        if accepted_len < step_gamma:
            rejection_events += 1
            if correction_id is not None and generated_target_tokens < max_new_tokens:
                current_ids = torch.cat(
                    [current_ids,
                     torch.tensor([[correction_id]], device=target_device)], dim=1
                )
                generated_target_tokens += 1

        if generated_target_tokens >= max_new_tokens:
            break

    generated_ids  = current_ids[0, prompt_len : prompt_len + max_new_tokens]
    generated_text = target_tok.decode(generated_ids, skip_special_tokens=True)
    full_text      = target_tok.decode(current_ids[0], skip_special_tokens=True)

    return {
        "prompt_id":        PROMPT_ID,
        "prompt_text":      prompt_text,
        "generated_text":   generated_text,
        "full_text":        full_text,
        "generated_tokens": generated_ids.shape[0],
        "accepted_tokens":  accepted_tokens,
        "proposed_tokens":  proposed_tokens,
        "rejection_events": rejection_events,
        "acceptance_rate":  (accepted_tokens / proposed_tokens) if proposed_tokens > 0 else 0.0,
    }

In [ ]:
def run_benchmark():
    # ── Baseline: pure target autoregressive ─────────────────────────────────
    messages = [{"role": "user", "content": PROMPT_TEXT}]
    formatted = target_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    input_ids = target_tok(formatted, return_tensors="pt")["input_ids"].to(
        next(target_model.parameters()).device
    )

    maybe_sync()
    t0 = time.perf_counter()
    with torch.no_grad():
        baseline_ids = target_model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )
    maybe_sync()
    baseline_time = time.perf_counter() - t0
    baseline_new  = baseline_ids.shape[1] - input_ids.shape[1]
    baseline_tps  = baseline_new / baseline_time

    baseline_text = target_tok.decode(baseline_ids[0, input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"BASELINE  | {baseline_new} tokens | {baseline_time:.2f}s | {baseline_tps:.1f} tok/s")

    # ── Speculative decoding ──────────────────────────────────────────────────
    maybe_sync()
    t1 = time.perf_counter()
    spec_result = speculative_decode(PROMPT_TEXT, max_new_tokens=MAX_NEW_TOKENS, gamma=GAMMA)
    maybe_sync()
    spec_time = time.perf_counter() - t1
    spec_tps  = spec_result["generated_tokens"] / spec_time

    print(f"SPEC DEC  | {spec_result['generated_tokens']} tokens | {spec_time:.2f}s | {spec_tps:.1f} tok/s")
    print(f"Generated Text: {(spec_result["generated_text"])}")
    print(f"  acceptance rate : {spec_result['acceptance_rate']:.2%}")
    print(f"  rejection events: {spec_result['rejection_events']}")
    print(f"  speedup         : {spec_tps / baseline_tps:.2f}x")
    print(f"{'='*60}\n")

run_benchmark()